# 21. What does the forgotten model actually predict?

A short, read-only check. It answers one question that the paired study never measured:
after a forgetting request, is the model **guessing** among the ten classes, or is it
giving the **same answer to every image**?

This matters because both look identical in the report. Each task is scored on 500
validation images, 50 in each of 10 classes. A model that answers one fixed class gets
exactly 50 right, which is exactly 10.0%. A model that truly guesses would land somewhere
near 10% but would move around, typically between about 7% and 13%. The paired study
reported exactly 10.0% in all six comparisons, which is the pattern of a fixed answer,
not of guessing.

Accuracy alone cannot tell these two apart. Counting how many images land in each class can.

### What this notebook does

1. Downloads one finished **source model**: a model that has already learned the eight
   earlier tasks, before any forgetting.
2. Applies **one forgetting request**, with the study's exact settings.
3. Stops every 20 steps and records three things: accuracy, which classes the model
   chooses, and how large the generated weights are.

### What it does not do

It trains nothing from scratch, writes nothing into the study, and cannot disturb a
future resume. It needs no new source model, because source models are kept in R2 while
branch models are deleted after verification. A GPU finishes in a few minutes; CPU works
but is slow.

### What the result cannot tell you

If the model turns out to give one fixed answer, that shows the classifier stopped
responding to its input. It does **not** show that the task's information was erased or
that it cannot be recovered. That is a separate experiment.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng
import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Choose which saved model to inspect

Pick one of the six completed comparisons. `SEED` chooses the run and `FORGET_TASK`
chooses which task is forgotten. Everything else matches the original study and should
be left alone, otherwise the numbers will not line up with the report.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

# Which saved model to inspect. These three values pick one finished source
# model out of the study. A source model is one that has already learned all
# eight earlier tasks, before any forgetting happened.
ORDER = "order_01"
SEED = 0                 # 0, 1, or 2: the three completed runs
FORGET_TASK = "3"        # "3" or "14": the task whose forgetting we watch

# How often to stop and measure during the 100 forgetting steps.
MEASURE_EVERY = 20

# Measuring every task costs about a second per task per stop. Keeping this on
# gives the comparison that matters: the forgotten task against the tasks that
# are supposed to survive. Turn it off only if you are in a hurry.
EVALUATE_ALL_TASKS = True

# Results are saved to R2, the same way notebook 20 saves its summary. A Colab
# runtime is deleted when the session ends and takes the local figure and
# tables with it, so a local-only copy is usually lost. This write goes into a
# separate "diagnostics" folder and never touches the study's own files, so it
# cannot affect a future resume. Set it to False to skip saving.
SAVE_TO_R2 = True

if FORGET_TASK not in ("3", "14"):
    raise ValueError("The completed runs only forgot task 3 or task 14.")
if SEED not in (0, 1, 2):
    raise ValueError("Choose SEED = 0, 1, or 2 from the saved study.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE, "" if DEVICE == "cuda" else "(this will be slow; an A100 finishes in a few minutes)")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Match the numerical settings the original runs used, so the generated
# weights come out identical rather than merely similar.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab
Secrets or environment variables. Their values are never printed or saved. This notebook
only reads the study; the one optional write goes to a separate diagnostics folder.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the saved study and the source model

The two checks below refuse to continue if the code revision or the class split differs
from the saved study. Without them the measurements would not be comparable with the
report.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())

# Refuse to run against a study built by different code or a different class
# split. Without these two checks the numbers below would not be comparable
# with the ones already in the report.
if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision. Use its recorded notebook revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

# cleanup_completed=False keeps this notebook from deleting anything in R2.
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)

SOURCE_ID = f"sources/{ORDER}/seed_{SEED}"
BRANCH_ID = f"branches/{ORDER}/seed_{SEED}/forget_{FORGET_TASK}/learn_{plan['new_tasks'][0]}"
source_job = next(job for job in jobs(plan) if job["id"] == SOURCE_ID)
branch_job = next(job for job in jobs(plan) if job["id"] == BRANCH_ID)

# Download the finished source model. Branch models were deleted after the
# study verified them, but source models are always kept, which is why this
# diagnostic needs no retraining.
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError("That source model is not finished in R2. Pick another seed.")

print("Source model:", SOURCE_ID)
print("Tasks it has already learned:", state["seen"])
print("Accuracy when the study finished training it:")
for task, value in state["previous"].items():
    print(f"   task {task:>3}: {value:5.1f}%")

## 5. Rebuild the model exactly as the study ran it

The saved file holds weights, not a working model, so the model is built first and the
weights are loaded into it. The random state is restored too, so the forgetting request
below draws the same noise the original run drew.

In [ ]:
# Rebuild the model the same way the study runner does, then load the saved
# weights into it. The request list is the branch's own: eight lessons, then
# the forgetting request, then learning task 15. We stop after forgetting.
requests = tuple(("learn", task) for task in plan["orders"][ORDER])
requests += tuple(tuple(operation) for operation in branch_job["operations"])
config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE, "requests": requests})

# Only the eight earlier tasks are needed. Task 15 is not learned here.
ORDER_TASKS = tuple(plan["orders"][ORDER])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=ORDER_TASKS, download=True)

target = build_target(config)
hypernet = HyperNetwork(target, config)
trainer = UnCLe(hypernet, config, target, tasks, progress=True)
checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
restore_rng(state["all_rng"])   # the same random state the real run had here

# Every task that was learned stays protected, except the one being forgotten.
# This single line is what the study runner does, so the run below is faithful.
PROTECTED = [task for task in state["seen"] if task != FORGET_TASK]
print("Forgetting task", FORGET_TASK)
print("Protected while forgetting:", PROTECTED)
print("Forgetting steps:", config.burn_in,
      "| learning rate:", config.forgetting_learning_rate,
      "| gamma:", config.gamma, "| beta:", config.beta)

## 6. Run the forgetting request and watch the predictions

Each printed row is one stop during the run. Read the last column: it counts how many of
the 500 validation images were placed in each of the ten classes. Watch whether those
counts collapse into a single number.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader

CLASSES = config.classes_per_task

def inspect(task):
    """Return accuracy and the count of images placed in each class.

    Accuracy alone cannot tell a model that guesses apart from a model that
    answers the same class every time: both can score 10%. The per-class
    counts separate them. A single bar of 500 means one constant answer.
    """
    weights = hypernet.weights_for(task)
    buffers = {name: value.clone() for name, value in trainer.task_buffers[task].items()}
    loader = DataLoader(tasks[task]["test"], batch_size=config.eval_batch_size)
    picked, correct, total = [], 0, 0
    for images, labels in loader:
        scores = functional_call(target, (weights, buffers),
                                 (images.to(trainer.device),), strict=True)
        choice = scores.argmax(1)
        picked.append(choice.cpu())
        correct += (choice == labels.to(trainer.device)).sum().item()
        total += labels.numel()
    counts = torch.bincount(torch.cat(picked), minlength=CLASSES).tolist()
    return 100.0 * correct / total, counts

records = []

def measure(step):
    """Record one row of evidence at this point in the forgetting run."""
    hypernet.eval()
    target.eval()
    # The size of the generated weights, as one number. If the forgetting step
    # is pushing the weights toward zero, this falls steadily.
    weight_size = hypernet.raw_for(FORGET_TASK).norm().item()
    accuracy, counts = inspect(FORGET_TASK)
    others = {}
    if EVALUATE_ALL_TASKS:
        for task in ORDER_TASKS:
            if task != FORGET_TASK:
                others[task], _ = inspect(task)
    row = {"step": step, "weight_size": weight_size, "accuracy": accuracy,
           "class_counts": counts, "distinct_classes": sum(1 for c in counts if c > 0),
           "other_tasks": others}
    records.append(row)
    print(f"step {step:>3} | accuracy {accuracy:5.1f}% | weight size {weight_size:9.2f} | "
          f"classes used {row['distinct_classes']:>2} | counts {counts}")
    return row

def on_step(record):
    """Called by forget() at step 0 and after every update. We sample it."""
    step = record["step"]
    if step == 0 or step % MEASURE_EVERY == 0:
        measure(step)

print("Measuring. Each row is one stop during the forgetting run.\n")
losses = trainer.forget(FORGET_TASK, PROTECTED, on_step=on_step)
if records[-1]["step"] != config.burn_in:
    measure(config.burn_in)
print("\nDone. Rows recorded:", len(records))

## 7. Read the result

In [ ]:
# The two questions this run was built to answer.
first = records[0]
last = records[-1]

print("QUESTION 1: is the forgotten model guessing, or giving one fixed answer?")
print(f"  before forgetting : accuracy {first['accuracy']:.1f}%, "
      f"used {first['distinct_classes']} of {CLASSES} classes")
print(f"  after forgetting  : accuracy {last['accuracy']:.1f}%, "
      f"used {last['distinct_classes']} of {CLASSES} classes")
if last["distinct_classes"] == 1:
    seat = max(range(CLASSES), key=lambda i: last["class_counts"][i])
    print(f"  ANSWER: one fixed answer. Every one of {sum(last['class_counts'])} "
          f"images was put in class {seat}.")
    print("  On a set with equal numbers of each class, that is exactly "
          f"{100.0 / CLASSES:.1f}% by arithmetic, not by chance.")
else:
    print(f"  ANSWER: predictions are still spread over {last['distinct_classes']} classes.")
    print("  The collapse explanation does not hold here. Share this output.")

print()
print("QUESTION 2: are the weights being driven to zero?")
print(f"  weight size before : {first['weight_size']:.2f}")
print(f"  weight size after  : {last['weight_size']:.2f}")
shrink = last["weight_size"] / first["weight_size"] if first["weight_size"] else float("nan")
print(f"  the weights ended at {shrink:.1%} of their original size")

if EVALUATE_ALL_TASKS:
    print()
    print("CONTROL: the same measuring code, on the tasks that should survive.")
    print("If these move around while the forgotten task sits still, the")
    print("measurement is working and the flatness belongs to forgetting alone.\n")
    header = "step |" + f"{FORGET_TASK:>7}*" + "".join(
        f"{task:>7}" for task in ORDER_TASKS if task != FORGET_TASK)
    print(header)
    print("-" * len(header))
    for row in records:
        line = f"{row['step']:>4} |{row['accuracy']:>7.1f} "
        line += "".join(f"{row['other_tasks'][task]:>7.1f}"
                        for task in ORDER_TASKS if task != FORGET_TASK)
        print(line)
    print("\n* the task being forgotten")
    moved = [t for t in ORDER_TASKS if t != FORGET_TASK
             and len({round(r["other_tasks"][t], 1) for r in records}) > 1]
    frozen = len({round(r["accuracy"], 1) for r in records[1:]}) == 1
    print(f"\nTasks whose accuracy changed during the run: {len(moved)} of {len(ORDER_TASKS) - 1}")
    print("Forgotten task frozen after its first drop:", frozen)

## 8. Chart

Three panels. Left: the forgotten task against the tasks that are supposed to survive.
Middle: the size of the generated weights. Right: where the 500 images are placed, before
and after.

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone

EXPORT = CACHE / "diagnostics" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT.mkdir(parents=True, exist_ok=True)

steps = [row["step"] for row in records]
figure, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# Left: every task's accuracy. The forgotten one is drawn thick and dark.
if EVALUATE_ALL_TASKS:
    for task in ORDER_TASKS:
        if task == FORGET_TASK:
            continue
        axes[0].plot(steps, [row["other_tasks"][task] for row in records],
                     color="#9bb7d4", linewidth=1.2)
axes[0].plot(steps, [row["accuracy"] for row in records],
             color="#b8322a", linewidth=2.6, marker="o", label=f"task {FORGET_TASK} (forgotten)")
axes[0].axhline(100.0 / CLASSES, color="black", linestyle=":", linewidth=1,
                label=f"{100.0/CLASSES:.0f}% line")
axes[0].set_xlabel("forgetting step"); axes[0].set_ylabel("accuracy (%)")
axes[0].set_title("Forgotten task against the protected ones")
axes[0].legend(fontsize=8)

# Middle: how big the generated weights are. Falling toward zero is the cause.
axes[1].plot(steps, [row["weight_size"] for row in records],
             color="#245d99", linewidth=2, marker="o")
axes[1].set_ylim(bottom=0)
axes[1].set_xlabel("forgetting step"); axes[1].set_ylabel("size of generated weights")
axes[1].set_title("Weights shrinking toward zero")

# Right: where the predictions go, before and after.
width = 0.4
positions = range(CLASSES)
axes[2].bar([p - width/2 for p in positions], first["class_counts"], width,
            label=f"step {first['step']}", color="#9bb7d4")
axes[2].bar([p + width/2 for p in positions], last["class_counts"], width,
            label=f"step {last['step']}", color="#b8322a")
axes[2].set_xlabel("predicted class"); axes[2].set_ylabel("images")
axes[2].set_title(f"Where task {FORGET_TASK}'s 500 images are placed")
axes[2].set_xticks(list(positions)); axes[2].legend(fontsize=8)

figure.suptitle(f"{STUDY_ID}  |  {ORDER}  seed {SEED}  |  forgetting task {FORGET_TASK}")
figure.tight_layout()
figure.savefig(EXPORT / "collapse_check.png", dpi=180)
figure.savefig(EXPORT / "collapse_check.svg")
plt.show()
print("Figure written to", EXPORT)

## 9. Save

Writes a JSON summary and a per-step CSV next to the figure, then copies all four files
to R2 under a diagnostics folder that is separate from the study's own files. Keep this on: a Colab
runtime is deleted when it ends, and the local copies go with it. Set `SAVE_TO_R2 = False`
in section 2 to skip the upload.

In [ ]:
import csv

summary = {
    "study_id": STUDY_ID, "code_version": CODE_VERSION, "order": ORDER, "seed": SEED,
    "forget_task": FORGET_TASK, "source_id": SOURCE_ID, "branch_id": BRANCH_ID,
    "device": DEVICE, "classes_per_task": CLASSES,
    "validation_images": sum(last["class_counts"]),
    "burn_in": config.burn_in, "gamma": config.gamma, "beta": config.beta,
    "forgetting_learning_rate": config.forgetting_learning_rate,
    "measure_every": MEASURE_EVERY, "evaluated_all_tasks": EVALUATE_ALL_TASKS,
    "distinct_classes_before": first["distinct_classes"],
    "distinct_classes_after": last["distinct_classes"],
    "accuracy_before": first["accuracy"], "accuracy_after": last["accuracy"],
    "weight_size_before": first["weight_size"], "weight_size_after": last["weight_size"],
    "forgetting_losses": losses, "records": records,
    "reading": ("One distinct class after forgetting means the model gives the same "
                "answer to every image. On a balanced set that is exactly chance by "
                "arithmetic. It does not show that the task's information was removed."),
}
(EXPORT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with (EXPORT / "steps.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["step", "accuracy", "weight_size", "distinct_classes"] + \
              [f"class_{i}" for i in range(CLASSES)] + \
              [f"task_{t}" for t in ORDER_TASKS if t != FORGET_TASK]
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    for row in records:
        line = {"step": row["step"], "accuracy": row["accuracy"],
                "weight_size": row["weight_size"], "distinct_classes": row["distinct_classes"]}
        line.update({f"class_{i}": count for i, count in enumerate(row["class_counts"])})
        line.update({f"task_{t}": value for t, value in row["other_tasks"].items()})
        writer.writerow(line)

print("Saved locally:")
for item in sorted(EXPORT.iterdir()):
    print("  ", item)

if SAVE_TO_R2:
    from tqdm.auto import tqdm
    # A separate folder. The study's own files are never touched.
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    for item in tqdm(sorted(EXPORT.iterdir()), desc="Upload diagnostics", unit="file"):
        store.upload(item, destination + "/" + item.name)
    print("Uploaded to", destination)
else:
    print("\nSAVE_TO_R2 is off. Set it to True to keep a copy in R2.")

## What to do with the output

**If the model ends up using one class**, the report's wording needs to change. Replace
"random guessing would score about 10%" with the real reason: the model answers one class
for every image, and the validation set holds 50 images of each of 10 classes, so one
fixed answer scores exactly 10.0%. Drop the phrase "chance level", because the model is
not at chance, it has stopped discriminating.

**If the model still spreads its predictions** across several classes while scoring 10.0%,
the collapse explanation is wrong and the exact 10.0% needs a different explanation. Share
the printed rows.

Either way, this run does not test whether the forgotten information is still recoverable.
Notebooks 16 and 17 set up that separate study.